# 03 — LP Optimization & Results

Goals:
- Define battery parameters
- Solve LP optimization for each day in the dataset
- Compare three scenarios: fixed tariff / dynamic no battery / dynamic + battery
- Visualize: price curve + charge/discharge schedule for a representative day
- Compute yearly savings in EUR

## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, value, LpStatus

# Battery parameters — adjust as needed
CAPACITY_KWH  = 10.0   # total usable capacity
MAX_POWER_KW  = 3.0    # max charge / discharge power per hour
EFFICIENCY    = 0.95   # round-trip efficiency (charge * discharge)
SOC_MIN       = 0.1    # minimum state of charge (fraction)
SOC_MAX       = 0.9    # maximum state of charge (fraction)

## 1. Load data

In [ ]:
df_prices = pd.read_csv('../../Data/prices_be.csv', parse_dates=['timestamp'])
# df_load  = pd.read_csv('../../Data/load_profile.csv', parse_dates=['timestamp'])  # add when available

# Placeholder: synthetic flat load profile (3500 kWh/year ≈ 0.40 kWh/hour average)
df_prices['load_kwh'] = 0.40

print(df_prices.head())

## 2. LP solver — single day

In [ ]:
def optimise_day(prices_kwh: list[float], load_kwh: list[float]) -> dict:
    """Solve LP for one 24-hour window. Returns cost and schedule."""
    T = range(24)
    prob = LpProblem('battery_dispatch', LpMinimize)

    charge    = [LpVariable(f'c_{t}', 0, MAX_POWER_KW) for t in T]
    discharge = [LpVariable(f'd_{t}', 0, MAX_POWER_KW) for t in T]
    soc       = [LpVariable(f's_{t}', SOC_MIN * CAPACITY_KWH,
                                      SOC_MAX * CAPACITY_KWH) for t in T]

    # Objective: minimise electricity cost
    prob += lpSum(prices_kwh[t] * (load_kwh[t] + charge[t] - discharge[t]) for t in T)

    # SOC dynamics
    prob += soc[0] == CAPACITY_KWH * 0.5  # start at 50%
    for t in range(1, 24):
        prob += soc[t] == soc[t-1] + EFFICIENCY * charge[t-1] - discharge[t-1] / EFFICIENCY

    prob.solve()

    return {
        'status'    : LpStatus[prob.status],
        'cost'      : value(prob.objective),
        'charge'    : [value(c) for c in charge],
        'discharge' : [value(d) for d in discharge],
        'soc'       : [value(s) for s in soc],
    }

## 3. Run optimisation over full dataset

> **TODO:** Implement daily loop once load profile data is available.

## 4. Compare scenarios

In [ ]:
# TODO: compute and compare
# cost_fixed     = average_price * total_load
# cost_dynamic   = sum(price[t] * load[t])
# cost_optimised = sum of daily LP objective values
#
# saving_vs_dynamic = cost_dynamic - cost_optimised
# saving_vs_fixed   = cost_fixed   - cost_optimised
pass

## 5. Visualisation — representative day

In [ ]:
# TODO: plot for a single day
# x-axis: hours 0–23
# twin axis: price curve (line) + charge/discharge bars + SOC line
pass